In [3]:
from pathlib import Path
import pandas as pd

RAW = Path("../data/raw")
competitions = pd.read_csv(RAW/"competitions.csv")
clubs = pd.read_csv(RAW/"clubs.csv")
players = pd.read_csv(RAW/"players.csv", parse_dates=["date_of_birth", "contract_expiration_date"])
valuations = pd.read_csv(RAW/"player_valuations.csv", parse_dates=["date"])
transfers = pd.read_csv(RAW/"transfers.csv", parse_dates=["transfer_date"])
salaries = pd.read_csv(RAW/"american_soccer_analysis_mls_salaries_players.csv")

for n, df in [("competition", competitions), ("clubs", clubs), ("players", players), ("valuations", valuations), ("transfers", transfers), ("salaries", salaries)]:
    print(n, df.shape)

competition (65, 11)
clubs (796, 17)
players (50149, 26)
valuations (656301, 6)
transfers (175165, 10)
salaries (14579, 8)


In [4]:
# MLS clubs
print(competitions[competitions.competition_id == "MLS1"].to_string())
mls_clubs = clubs[clubs.domestic_competition_id == "MLS1"]
print(len(mls_clubs))
print(mls_clubs[["club_id", "name", "last_season"]].sort_values("name").to_string())

   competition_id     competition_code                 name    sub_type             type  country_id   country_name domestic_league_code confederation  total_clubs                                                                             url
38           MLS1  major-league-soccer  major-league-soccer  first_tier  domestic_league         184  United States                 MLS1       amerika         30.0  https://www.transfermarkt.co.uk/major-league-soccer/startseite/wettbewerb/MLS1
30
     club_id                                name  last_season
556    51663        Atlanta United Football Club         2025
691    72309                           Austin FC         2025
708    78435             Charlotte Football Club         2025
507      432            Chicago Fire Soccer Club         2025
665    69261  Club Internacional de Fútbol Miami         2025
479     4078               Club de Foot Montréal         2025
104     1247                     Colorado Rapids         2025
719      813 

In [5]:
# Valuations: MLS rows and unique players per year
mv = valuations[valuations.player_club_domestic_competition_id == "MLS1"]
print(len(mv))
print(mv.groupby(mv.date.dt.year).agg(rows=("player_id", "size"), players=("player_id", "nunique")))

9351
      rows  players
date               
2004     1        1
2006     1        1
2007     1        1
2008     8        5
2009    17        8
2010    16        7
2011    19       12
2012    26       16
2013    28       17
2014    28       15
2015    70       54
2016   217       91
2017   264      136
2018   383      180
2019   460      225
2020   924      330
2021   833      399
2022   939      457
2023  1150      555
2024  1312      622
2025  1686      745
2026   968      806


In [6]:
# Transfers touching MLS clubs, 2016-2025
ids = set(mls_clubs.club_id)
tm = transfers[(transfers.from_club_id.isin(ids) | transfers.to_club_id.isin(ids))
        & transfers.transfer_date.between("2016-01-01", "2025-12-31")]
print(len(tm))
print(tm.groupby(tm.transfer_date.dt.year).size())
print(tm.transfer_fee.isna().sum(), (tm.transfer_fee == 0).sum())
print(tm[["from_club_name", "to_club_name", "transfer_fee"]].head(15).to_string())

3151
transfer_date
2016     98
2017    157
2018    182
2019    281
2020    237
2021    320
2022    379
2023    384
2024    493
2025    620
dtype: int64
581 1857
       from_club_name   to_club_name  transfer_fee
8301          Toronto  Vasco da Gama           0.0
8309            Miami       Atlético           0.0
8313        Charlotte     Olympiacos           0.0
8325        San Diego  Celta de Vigo           0.0
8329        Vancouver         Chivas           0.0
8355        San Diego          Pafos           0.0
8356       Cincinnati        Udinese           0.0
8377          Chicago          Porto           0.0
8378            Atlas       Montréal           0.0
8395  Louisville City      Salt Lake           0.0
8397    Górnik Zabrze      Minnesota           0.0
8401            Miami   Boca Juniors           0.0
8407   St. Louis CITY   Slavia Praha           0.0
8409       Cincinnati  Middlesbrough           0.0
8412      New England      FC Nantes           0.0


In [7]:
# ASA salaries
print(salaries.Season.value_counts().sort_index())
print(salaries.Team.nunique(), salaries.Player.nunique())
print(salaries.head(10).to_string())

Season
2016    1128
2017    1267
2018    1345
2019    1386
2020     771
2021    1636
2022    1675
2023    1784
2024    1761
2025    1826
Name: count, dtype: int64
30 2677
   Unnamed: 0           Player Team  Season Position  Base Salary Guaranteed Compensation        Date
0         NaN     Lionel Messi  MIA    2023        M  $12,000,000             $20,446,668  2023-09-15
1         NaN     Lionel Messi  MIA    2024        F  $12,000,000             $20,446,668  2024-04-25
2         NaN     Lionel Messi  MIA    2024        F  $12,000,000             $20,446,668  2024-09-13
3         NaN     Lionel Messi  MIA    2025        F  $12,000,000             $20,446,668  2025-05-23
4         NaN     Lionel Messi  MIA    2025        F  $12,000,000             $20,446,668  2025-10-01
5         NaN  Lorenzo Insigne  TOR    2023        M   $7,500,000             $15,400,000  2023-09-15
6         NaN  Lorenzo Insigne  TOR    2024        F   $7,500,000             $15,400,000  2024-04-25
7         NaN